In [10]:
import pandas as pd
import numpy as np

# 1. 데이터 로드
store_info = pd.read_csv("./data2/store_info.csv")
daily_cashflow = pd.read_csv("./data2/daily_cashflow.csv")
fixed_cost = pd.read_csv("./data2/fixed_cost.csv")
loan_info = pd.read_csv("./data2/loan_info.csv")

# 날짜형 변환
daily_cashflow["date"] = pd.to_datetime(daily_cashflow["date"])
fixed_cost["month"] = pd.to_datetime(fixed_cost["month"] + "-01")
loan_info["loan_start_date"] = pd.to_datetime(loan_info["loan_start_date"])

# 정렬
daily_cashflow = daily_cashflow.sort_values(["store_id", "date"]).reset_index(drop=True)
fixed_cost = fixed_cost.sort_values(["store_id", "month"]).reset_index(drop=True)
loan_info = loan_info.sort_values(["store_id", "loan_start_date"]).reset_index(drop=True)

# =========================================================
# 2. 기본 전처리
# =========================================================

# 일별 총 지출
daily_cashflow["daily_expense"] = (
    daily_cashflow["platform_fee"]
    + daily_cashflow["ingredient_cost"]
    + daily_cashflow["labor_cost"]
    + daily_cashflow["utilities_cost"]
    + daily_cashflow["marketing_cost"]
    + daily_cashflow["other_expense"]
    + daily_cashflow["loan_repayment"]
)

# 월 컬럼 생성
daily_cashflow["month"] = daily_cashflow["date"].values.astype("datetime64[M]")

# 월별 매출
monthly_sales = (
    daily_cashflow.groupby(["store_id", "month"], as_index=False)["total_sales"]
    .sum()
    .rename(columns={"total_sales": "monthly_sales"})
)

# 월별 대출 상환액
monthly_loan = (
    loan_info.groupby("store_id", as_index=False)["monthly_payment"]
    .sum()
    .rename(columns={"monthly_payment": "total_monthly_payment"})
)

# 고정비 증가율 계산
fixed_cost["fixed_cost_3m_ago"] = fixed_cost.groupby("store_id")["fixed_cost_total"].shift(3)

fixed_cost["fixed_cost_growth_rate"] = np.where(
    fixed_cost["fixed_cost_3m_ago"] > 0,
    (fixed_cost["fixed_cost_total"] - fixed_cost["fixed_cost_3m_ago"]) / fixed_cost["fixed_cost_3m_ago"],
    0.0
)

fixed_cost["fixed_cost_growth_rate"] = (
    fixed_cost["fixed_cost_growth_rate"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .clip(0, 1)
)

# =========================================================
# 3. 데이터 결합
# =========================================================

df = daily_cashflow.merge(
    fixed_cost[["store_id", "month", "fixed_cost_total", "fixed_cost_growth_rate"]],
    on=["store_id", "month"],
    how="left"
)

df = df.merge(monthly_sales, on=["store_id", "month"], how="left")
df = df.merge(monthly_loan, on="store_id", how="left")

# 결측값 처리
df["fixed_cost_total"] = df["fixed_cost_total"].fillna(df["fixed_cost_total"].median())
df["fixed_cost_growth_rate"] = df["fixed_cost_growth_rate"].fillna(0)
df["monthly_sales"] = df["monthly_sales"].fillna(0)
df["total_monthly_payment"] = df["total_monthly_payment"].fillna(0)

df = df.sort_values(["store_id", "date"]).reset_index(drop=True)

# =========================================================
# 4. 파생 변수 생성
# =========================================================

g = df.groupby("store_id")

# 7일 평균 매출
df["sales_7d_avg"] = (
    g["total_sales"]
    .rolling(7, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# 30일 평균 매출
df["sales_30d_avg"] = (
    g["total_sales"]
    .rolling(30, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# 30일 평균 매출 / 지출
df["avg_daily_sales_30"] = (
    g["total_sales"]
    .rolling(30, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

df["avg_daily_expense_30"] = (
    g["daily_expense"]
    .rolling(30, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# 매출 추세 계수
df["sales_trend_factor"] = np.where(
    df["sales_30d_avg"] > 0,
    df["sales_7d_avg"] / df["sales_30d_avg"],
    1.0
)

df["sales_trend_factor"] = (
    df["sales_trend_factor"]
    .replace([np.inf, -np.inf], 1.0)
    .fillna(1.0)
    .clip(0.5, 1.5)
)

# 최근 30일 매출
df["curr_30d_sales"] = (
    g["total_sales"]
    .rolling(30, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

df["prev_30d_sales"] = g["curr_30d_sales"].shift(30)

# 매출 감소율
df["sales_decline_rate"] = np.where(
    df["prev_30d_sales"] > 0,
    (df["prev_30d_sales"] - df["curr_30d_sales"]) / df["prev_30d_sales"],
    0.0
)

df["sales_decline_rate"] = (
    df["sales_decline_rate"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .clip(0, 1)
)

# 현금 runway
daily_burn = (df["avg_daily_expense_30"] - df["avg_daily_sales_30"]).clip(lower=1)

df["cash_runway_days"] = df["closing_balance"] / daily_burn
df["cash_runway_days"] = df["cash_runway_days"].replace([np.inf, -np.inf], 9999).fillna(9999)

# 고정비 대비 잔액
df["fixed_cost_ratio"] = np.where(
    df["closing_balance"] > 0,
    df["fixed_cost_total"] / df["closing_balance"],
    999
)

df["fixed_cost_ratio"] = df["fixed_cost_ratio"].replace([np.inf, -np.inf], 999).fillna(999)

# 대출 부담률
df["loan_burden_ratio"] = np.where(
    df["monthly_sales"] > 0,
    df["total_monthly_payment"] / df["monthly_sales"],
    0.0
)

df["loan_burden_ratio"] = (
    df["loan_burden_ratio"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .clip(0, 1)
)

# 90일 전 잔액
df["balance_90d_ago"] = g["closing_balance"].shift(90)

df["cash_decline_rate_90d"] = np.where(
    df["balance_90d_ago"] > 0,
    (df["balance_90d_ago"] - df["closing_balance"]) / df["balance_90d_ago"],
    0.0
)

df["cash_decline_rate_90d"] = (
    df["cash_decline_rate_90d"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .clip(0, 1)
)

# =========================================================
# 5. 월말 잔액 예측
# =========================================================

df["month_end_date"] = df["date"] + pd.offsets.MonthEnd(0)
df["days_remaining"] = (df["month_end_date"] - df["date"]).dt.days

df["predicted_end_balance"] = (
    df["closing_balance"]
    + (df["avg_daily_sales_30"] * df["sales_trend_factor"] * df["days_remaining"])
    - (df["avg_daily_expense_30"] * df["days_remaining"])
)

df["predicted_runway_days"] = np.where(
    df["avg_daily_expense_30"] > 0,
    df["predicted_end_balance"] / df["avg_daily_expense_30"],
    999
)

df["predicted_runway_days"] = df["predicted_runway_days"].replace([np.inf, -np.inf], 999).fillna(999)

# =========================================================
# 6. 실제 월말 잔액
# =========================================================

actual_month_end = (
    df.sort_values(["store_id", "date"])
    .groupby(["store_id", "month"])
    .tail(1)[["store_id", "month", "closing_balance"]]
    .rename(columns={"closing_balance": "actual_end_balance"})
)

df = df.merge(actual_month_end, on=["store_id", "month"], how="left")

# =========================================================
# 7. 단기 위험 점수
# =========================================================

df["short_fixed_cost_risk"] = np.where(
    df["closing_balance"] > 0,
    (df["fixed_cost_total"] / df["closing_balance"]) * 100,
    100
)

df["short_fixed_cost_risk"] = (
    df["short_fixed_cost_risk"]
    .replace([np.inf, -np.inf], 100)
    .fillna(100)
    .clip(0, 100)
)

df["short_sales_decline_risk"] = (df["sales_decline_rate"] * 100).clip(0, 100)

def score_runway_risk(runway_days):
    if pd.isna(runway_days):
        return 50
    elif runway_days <= 0:
        return 100
    elif runway_days <= 3:
        return 90
    elif runway_days <= 7:
        return 75
    elif runway_days <= 14:
        return 55
    elif runway_days <= 30:
        return 30
    else:
        return 10

df["short_runway_risk"] = df["predicted_runway_days"].apply(score_runway_risk)

df["short_risk"] = (
    df["short_fixed_cost_risk"] * 0.4
    + df["short_sales_decline_risk"] * 0.3
    + df["short_runway_risk"] * 0.3
).clip(0, 100)

# =========================================================
# 8. 중기 위험 점수
# =========================================================

df["mid_cash_decline_risk"] = (df["cash_decline_rate_90d"] * 100).clip(0, 100)
df["mid_fixed_cost_growth_risk"] = (df["fixed_cost_growth_rate"] * 100).clip(0, 100)
df["mid_loan_burden_risk"] = (df["loan_burden_ratio"] * 100).clip(0, 100)

df["mid_risk"] = (
    df["mid_cash_decline_risk"] * 0.4
    + df["mid_fixed_cost_growth_risk"] * 0.3
    + df["mid_loan_burden_risk"] * 0.3
).clip(0, 100)

# =========================================================
# 9. 종합 위험 점수
# =========================================================

df["total_risk"] = (
    df["short_risk"] * 0.6
    + df["mid_risk"] * 0.4
).clip(0, 100)

# =========================================================
# 10. 위험 레벨
# =========================================================

def classify_risk_level(score):
    if score < 25:
        return "안정"
    elif score < 50:
        return "주의"
    elif score < 75:
        return "위험"
    else:
        return "매우 위험"

df["risk_level"] = df["total_risk"].apply(classify_risk_level)

# =========================================================
# 11. 최종 결과
# =========================================================

result = df[
    [
        "store_id",
        "date",
        "closing_balance",
        "predicted_end_balance",
        "short_risk",
        "mid_risk",
        "total_risk",
        "risk_level",
        "actual_end_balance",
    ]
].copy()

for col in [
    "closing_balance",
    "predicted_end_balance",
    "short_risk",
    "mid_risk",
    "total_risk",
    "actual_end_balance",
]:
    result[col] = result[col].round(2)

result.to_csv("risk_prediction_result.csv", index=False, encoding="utf-8-sig")

print(result.head(10))
print(result.columns.tolist())

  store_id       date  closing_balance  predicted_end_balance  short_risk  \
0     S001 2025-01-05          4241482             8291398.00       56.04   
1     S001 2025-01-07          2107282             5173210.00       62.50   
2     S001 2025-01-21          4877956             5555239.33       56.88   
3     S001 2025-01-24          2274719             2773078.75       62.50   
4     S001 2025-02-01          2098176             4712634.60       62.50   
5     S001 2025-02-03          2078070             4019740.83       62.50   
6     S001 2025-02-06          2877356             4458090.57       62.50   
7     S001 2025-02-08          4293566             5471529.57       55.56   
8     S001 2025-02-09          5202680             6301844.78       48.73   
9     S001 2025-02-09          3306022             3956012.27       62.50   

   mid_risk  total_risk risk_level  actual_end_balance  
0      2.16       34.49         주의             2274719  
1      2.16       38.36         주의    

In [11]:
result["risk_level"].value_counts()

risk_level
주의    4598
안정       6
Name: count, dtype: int64